<a href="https://colab.research.google.com/github/Boulder1-kihara/gemma4-RAG-study-helper/blob/main/study_helper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!pip install langchain langchain-community langchain-chroma sentence-transformers
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers duckduckgo-search langchain-google-genai pypdf


In [13]:
# 1. Install the missing zstd extractor
!apt-get install -y zstd

# 2. Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the local server in the background
!nohup ollama serve > ollama.log 2>&1 &

# 4. Pull the Gemma 4 model
!ollama pull gemma4

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [14]:
# 1. Install the system OCR tool so the agent can read text inside images
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils

# 2. Install unstructured with the "all-docs" flag to handle Word, PPT, etc.
!pip install -qU "unstructured[all-docs]" langchain-community

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [16]:
!pip install -qU "unstructured[all-docs]" langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers duckduckgo-search pypdf langchain-huggingface langchain-ollama

In [20]:
!apt-get update
!apt-get install -y libreoffice
!soffice --version

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,480 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,844 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRel

In [24]:
import os
import shutil
from langchain_community.vectorstores import Chroma

# Let's put the Deep Intel brain directly in your Google Drive so it is permanent
persist_dir = "/content/drive/MyDrive/Deep_Intel_Brain"

# Clear out the directory if a broken version exists
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)

print(f"Saving {len(chunks)} chunks permanently to Google Drive...")

# Save it using the chunks and embeddings already in your RAM
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Success! Your agent's brain is now safely stored on Google Drive.")

Saving 1065 chunks permanently to Google Drive...
Success! Your agent's brain is now safely stored on Google Drive.


In [25]:
!pip install -U ddgs

from langchain_core.tools import create_retriever_tool
from langchain_community.tools import DuckDuckGoSearchRun

# Tool 1: The Local Notes Search
notes_tool = create_retriever_tool(
    retriever,
    name="university_notes_search",
    description="Search for information, definitions, and concepts from the user's official class notes. Always use this tool first."
)

# Tool 2: The Web Fallback
web_search = DuckDuckGoSearchRun()
web_tool = web_search

tools = [notes_tool, web_tool]

In [26]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# 1. Initialize the LLM with the CURRENT active model version
llm = ChatOllama(
    model="gemma4",
    temperature=0
)

# 2. Define your system instructions (Notice the spaces added at the end of the lines!)
system_prompt = (
    "You were created by Abel, an engineer at Deep Intel. Your name is 'University Study Helper'. "
    "You are a helpful university tutor. Answer the student's past paper questions. "
    "First, search the 'university_notes_search' tool. If the answer is complete, use it. "
    "If the notes do not contain the answer or are missing details, use the web search tool to find the rest."
)

# 3. Create the LangGraph Agent
agent_executor = create_react_agent(llm, tools, prompt=system_prompt)

/tmp/ipykernel_8255/1486121914.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=system_prompt)


In [28]:
# Wake up the Ollama background server
!nohup ollama serve > ollama.log 2>&1 &

# Give it a second to boot, then verify Gemma 4 is ready
!sleep 3
!ollama list

NAME             ID              SIZE      MODIFIED       
gemma4:latest    c6eb396dbd59    9.6 GB    47 minutes ago    


In [29]:
from IPython.display import display, Markdown

# The question from your past paper
question = input("Type your Kirinyaga past paper question here: ")

print("\nSearching notes and firing up Gemma 4 locally...")

# Invoke the local agent
response = agent_executor.invoke({"messages": [("human", question)]})

# Extract the text and render it beautifully
raw_content = response["messages"][-1].content
if isinstance(raw_content, list):
    final_text = raw_content[0].get("text", "")
else:
    final_text = raw_content

display(Markdown(final_text))

Type your Kirinyaga past paper question here: define mbile computing'

Searching notes and firing up Gemma 4 locally...


Mobile computing is a broad term that describes the ability to use technology while moving, as opposed to portable computers, which are only practical for use in a stationary setup.

In essence, it refers to the set of IT technologies, products, services, and operational strategies that allow users to access computation, information, and related resources while mobile.

Here is a more detailed breakdown of the concept:

### Core Concept
*   **Definition:** Mobile computing is the technology that enables the transmission of data, voice, and video via a computer or any other wireless-enabled device *without* needing to be connected to a fixed physical link.
*   **Scope:** The term "mobile" most commonly refers to access *in motion*, meaning it is unrestricted by a specific geographic location.

### Key Technologies Involved
Mobile computing relies on various wireless networks, including:
*   Infrared (IR)
*   Bluetooth
*   W-LANs (Wireless Local Area Networks)
*   Cellular networks
*   W-Packet Data networks, etc.

### Forms of Computing
The concept encompasses several related forms:

1.  **Wireless Computing:** Transferring data or information between devices that are not physically connected, relying on a "wireless network connection."
2.  **Nomadic Computing:** Using mobile technology to connect to the global Internet or access specific data resources while moving from one place to another.
3.  **Ubiquitous Computing:** An advanced concept where computing is designed to appear and function everywhere and anywhere.
4.  **Mobile Cloud Computing:** This is a significant modern aspect. It refers to an infrastructure where both **data storage and data processing happen *outside* of the mobile device** (in the cloud). Instead of running everything on the phone, the power and storage are centralized in powerful cloud platforms, which are then accessed by the mobile device via an app or web browser.

In summary, mobile computing is about **unrestricted access to computing power and data** through wireless means, with modern advancements pushing this capability into the cloud and making it seem omnipresent (ubiquitous).